In [18]:
import effspm

# Print all available functions exported by pybind11
print("Available effspm functions:")
print(dir(effspm))

Available effspm functions:
['BTMiner', 'HTMiner', 'LargeBTMiner', 'LargeHTMiner', 'LargePrefixProjection', 'PrefixProjection', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_effspm']


In [19]:
%pip install pandas


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
import effspm
import pandas as pd
import os

# Update dataset path to point to a file inside your 'datasets' folder
DATASET_PATH = os.path.join("datasets", "your_dataset_file.txt")  # <-- Update filename

# Support thresholds from Table I
THRESHOLDS = [0.01, 0.05, 0.10, 0.15, 0.20]

# Algorithms to test
ALGORITHMS = ["htminer", "btminer", "prefixprojection", "largehm", "largebm", "largepp"]

results = []

for t in THRESHOLDS:
    for algo_name in ALGORITHMS:
        try:
            algo_func = getattr(effspm, algo_name)
            patterns = algo_func(DATASET_PATH, t)
            
            count = len(patterns) if isinstance(patterns, list) else patterns
            results.append({"Threshold": t, "Algorithm": algo_name, "Pattern Count": count, "Status": "Success"})
        except AttributeError:
            results.append({"Threshold": t, "Algorithm": algo_name, "Pattern Count": None, "Status": "Function Not Found"})
        except Exception as e:
            results.append({"Threshold": t, "Algorithm": algo_name, "Pattern Count": None, "Status": f"Error: {e}"})

# Render table in Jupyter
df_results = pd.DataFrame(results)
df_results


,Threshold,Algorithm,Pattern Count,Status
0,0.01,htminer,None,Function Not Found
1,0.01,btminer,None,Function Not Found
2,0.01,prefixprojection,None,Function Not Found
3,0.01,largehm,None,Function Not Found
4,0.01,largebm,None,Function Not Found
5,0.01,largepp,None,Function Not Found
6,0.05,htminer,None,Function Not Found
7,0.05,btminer,None,Function Not Found
8,0.05,prefixprojection,None,Function Not Found
9,0.05,largehm,None,Function Not Found


In [21]:
import os
import time
from effspm import HTMiner  # Adjust based on your package structure

DATASET_DIR = "/Users/sandeep/Documents/library/effspm/datasets"
test_cases = [
    {"file": "MSNBC.txt", "min_sup": 0.01},
    {"file": "kosarak.txt", "min_sup": 0.005}
]

for test in test_cases:
    path = os.path.join(DATASET_DIR, test["file"])
    if not os.path.exists(path):
        print(f"File not found: {path}")
        continue

    print(f"\n--- Testing {test['file']} (min_sup={test['min_sup']}) ---")
    start_time = time.time()
    
    # Initialize and execute miner
    miner = HTMiner(input_path=path, min_support=test["min_sup"])
    miner.run()
    
    elapsed = time.time() - start_time
    print(f"Status: Completed in {elapsed:.3f} seconds")
    print(f"Patterns Found: {miner.get_pattern_count()}")


--- Testing MSNBC.txt (min_sup=0.01) ---


TypeError: HTMiner(): incompatible function arguments. The following argument types are supported:
    1. (data: object, minsup: typing.SupportsFloat | typing.SupportsIndex = 0.01, time_limit: typing.SupportsInt | typing.SupportsIndex = 36000, preproc: bool = False, use_dic: bool = False, verbose: bool = False, out_file: str = '') -> dict

Invoked with: kwargs: input_path='/Users/sandeep/Documents/library/effspm/datasets/MSNBC.txt', min_support=0.01

In [ ]:
# Option 1: Positional arguments (Recommended)
miner = HTMiner(path, test["min_sup"])

# Option 2: Exact keyword arguments
miner = HTMiner(data=path, minsup=test["min_sup"])

In [ ]:
import os
import time
from effspm import HTMiner  # Adjust based on your package structure

DATASET_DIR = "/Users/sandeep/Documents/library/effspm/datasets"
test_cases = [
    {"file": "MSNBC.txt", "min_sup": 0.01},
    {"file": "kosarak.txt", "min_sup": 0.005}
]

for test in test_cases:
    path = os.path.join(DATASET_DIR, test["file"])
    if not os.path.exists(path):
        print(f"File not found: {path}")
        continue

    print(f"\n--- Testing {test['file']} (minsup={test['min_sup']}) ---")
    start_time = time.time()
    
    # Updated call using positional arguments matching pybind11 / C++ signature
    results = HTMiner(path, test["min_sup"])
    
    elapsed = time.time() - start_time
    print(f"Status: Completed in {elapsed:.3f} seconds")
    
    # If HTMiner executes immediately upon instantiation and returns a dict:
    if isinstance(results, dict):
        print(f"Total Patterns Mined: {len(results)}")

In [ ]:
import os
import pandas as pd
from effspm import HTMiner  # Adjust imports for specific methods if applicable

DATASET_DIR = "/Users/sandeep/Documents/library/effspm/datasets"

# Define your testing grid based on Table 3
test_grid = [
    {"dataset": "MSNBC.txt", "thresholds": [0.05, 0.01, 0.005]},
    {"dataset": "kosarak.txt", "thresholds": [0.01, 0.005, 0.001]}
]

records = []

for item in test_grid:
    ds_name = item["dataset"]
    ds_path = os.path.join(DATASET_DIR, ds_name)
    
    for minsup in item["thresholds"]:
        # 1. Run via effspm Python overlay
        try:
            py_res = HTMiner(ds_path, minsup)
            py_count = len(py_res) if isinstance(py_res, dict) else None
        except Exception as e:
            py_count = f"Error: {e}"
            
        # 2. Add entry (you will fill cpp_count from binary output or subprocess call)
        records.append({
            "Dataset": ds_name,
            "Minsup": minsup,
            "Python_effspm_Count": py_count,
            "Cpp_HTMiner_Count": "Pending C++ Run",
            "Table3_GroundTruth": "Extract from Paper"
        })

df = pd.DataFrame(records)
print(df)

In [ ]:
results = HTMiner("/Users/sandeep/Documents/library/effspm/datasets/MSNBC.txt", 0.01)

print("Dictionary Keys:", results.keys())
print("Sample Output:", results)

In [ ]:
# Assuming results is a dict containing a nested dict/list of patterns
if isinstance(results, dict):
    if "patterns" in results:
        py_count = len(results["patterns"])
    else:
        # If the dict maps pattern_string -> support_value directly
        py_count = len(results)

In [ ]:
import os
from effspm import HTMiner

DATASET_DIR = "/Users/sandeep/Documents/library/effspm/datasets"
test_path = os.path.join(DATASET_DIR, "MSNBC.txt")

# Run HTMiner on MSNBC
results = HTMiner(test_path, 0.01)

# Inspect output structure
print("Type of output:", type(results))
print("Dictionary keys:", results.keys())
print("\nSample preview of values:")
for key in results.keys():
    val = results[key]
    print(f"  Key '{key}': type={type(val)}, value/length={len(val) if hasattr(val, '__len__') else val}")

In [ ]:
import os
import pandas as pd
from effspm import HTMiner

DATASET_DIR = "/Users/sandeep/Documents/library/effspm/datasets"

# Test grid matching Table 3 thresholds
test_grid = [
    {"dataset": "MSNBC.txt", "thresholds": [0.05, 0.01, 0.005]},
    {"dataset": "kosarak.txt", "thresholds": [0.01, 0.005, 0.001]}
]

records = []

for item in test_grid:
    ds_name = item["dataset"]
    ds_path = os.path.join(DATASET_DIR, ds_name)
    
    for minsup in item["thresholds"]:
        try:
            res = HTMiner(ds_path, minsup)
            # Correctly target the 'patterns' key in the returned dictionary
            py_count = len(res.get('patterns', []))
            exec_time = round(res.get('time', 0), 4)
        except Exception as e:
            py_count = f"Error: {e}"
            exec_time = None
            
        records.append({
            "Dataset": ds_name,
            "Minsup": minsup,
            "Python_effspm_Count": py_count,
            "Python_Time_Sec": exec_time,
            "Cpp_HTMiner_Count": "Pending",
            "Table3_Count": "Pending"
        })

df = pd.DataFrame(records)
print(df.to_string(index=False))

In [ ]:
import subprocess
import os
import re

def run_local_cpp_htminer(file_name, minsup):
    # Set path directly to your local directory
    cpp_dir = "/Users/sandeep/Documents/library/HTMiner/HTMiner"
    
    # Define exact command targeting your local compiled binary
    cmd = [
        os.path.join(cpp_dir, "HTMiner"),
        "-folder", "datasets/",
        "-file", file_name,
        "-thr", str(minsup)
    ]
    
    # Execute the local C++ binary in its own working directory
    process = subprocess.run(
        cmd, 
        capture_output=True, 
        text=True, 
        cwd=cpp_dir
    )
    
    # Parse output printed by C++ stdout
    output = process.stdout
    match = re.search(r"Found a total of (\d+) patterns", output)
    
    if match:
        return int(match.group(1))
    else:
        print("C++ Execution Output:\n", output)
        return None

# Test directly in Jupyter:
msnbc_count = run_local_cpp_htminer("MSNBC", 0.05)
print("Pattern Count from local C++ build:", msnbc_count)

In [ ]:
import os
import re
import subprocess
import pandas as pd
from effspm import HTMiner as PythonHTMiner

# 1. Configuration Paths
CPP_DIR = "/Users/sandeep/Documents/library/HTMiner/HTMiner"
DATASET_DIR = "/Users/sandeep/Documents/library/effspm/datasets"

# Paper Table 3 Ground Truth Mapping
PAPER_GROUND_TRUTH = {
    ("MSNBC.txt", 0.050): 13,
    ("MSNBC.txt", 0.010): 254,
    ("MSNBC.txt", 0.005): 828,
    ("kosarak.txt", 0.010): 329,
    ("kosarak.txt", 0.005): 1462,
    ("kosarak.txt", 0.001): 758141,
}

test_grid = [
    {"file": "MSNBC.txt", "name_no_ext": "MSNBC", "thresholds": [0.050, 0.010, 0.005]},
    {"file": "kosarak.txt", "name_no_ext": "kosarak", "thresholds": [0.010, 0.005, 0.001]}
]

# Helper function to invoke local native C++ executable
def run_native_cpp(file_no_ext, minsup):
    cmd = [os.path.join(CPP_DIR, "HTMiner"), "-folder", "datasets/", "-file", file_no_ext, "-thr", str(minsup)]
    process = subprocess.run(cmd, capture_output=True, text=True, cwd=CPP_DIR)
    match = re.search(r"Found a total of (\d+) patterns", process.stdout)
    return int(match.group(1)) if match else None

# 2. Benchmark Execution Loop
comparison_data = []

for test in test_grid:
    ds_path = os.path.join(DATASET_DIR, test["file"])
    for minsup in test["thresholds"]:
        
        # Source 1: Python Wrapper (effspm)
        py_res = PythonHTMiner(ds_path, minsup)
        py_count = len(py_res['patterns'])
        py_time = round(py_res['time'], 4)
        
        # Source 2: Standalone Native C++ Binary
        cpp_count = run_native_cpp(test["name_no_ext"], minsup)
        
        # Source 3: Paper Ground Truth (Table 3)
        paper_count = PAPER_GROUND_TRUTH.get((test["file"], minsup))
        
        # Parity Check Logic
        parity = "PASS" if (py_count == cpp_count == paper_count) else "FAIL"
        
        comparison_data.append({
            "Dataset": test["file"],
            "Minsup": minsup,
            "Python_effspm_Count": py_count,
            "Python_Time_Sec": py_time,
            "Cpp_HTMiner_Count": cpp_count,
            "Table3_Paper_Count": paper_count,
            "Parity_Status": parity
        })

# 3. Output Full Comparison Table
df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

In [ ]:
import os
import pandas as pd
import effspm

DATASET_DIR = "/Users/sandeep/Documents/library/effspm/datasets"

# Ground truth numbers from Table 3 of the JMLR paper
paper_truth = {
    ("MSNBC.txt", 0.05): 13,
    ("MSNBC.txt", 0.01): 254,
    ("MSNBC.txt", 0.005): 828,
    ("kosarak.txt", 0.01): 329,
    ("kosarak.txt", 0.005): 1462,
    ("kosarak.txt", 0.001): 758141,
}

algorithms = ["HTMiner", "BTMiner", "PrefixProjection"]

rows = []

for (dataset, minsup), paper_count in paper_truth.items():
    ds_path = os.path.join(DATASET_DIR, dataset)
    row = {
        "Dataset": dataset,
        "Minsup": minsup,
        "Paper (Table 3)": paper_count
    }
    
    for algo in algorithms:
        try:
            miner_func = getattr(effspm, algo)
            res = miner_func(ds_path, minsup)
            # Extract count using the key fix
            count = len(res['patterns']) if isinstance(res, dict) and 'patterns' in res else len(res)
            row[f"{algo} Count"] = count
        except Exception as e:
            row[f"{algo} Count"] = f"Error: {e}"
            
    # Check if all tested algorithms match the paper ground truth
    algo_counts = [row[f"{algo} Count"] for algo in algorithms]
    row["All Match?"] = "PASS" if all(c == paper_count for c in algo_counts) else "FAIL"
    
    rows.append(row)

# Display final clear side-by-side comparison
df_comparison = pd.DataFrame(rows)
df_comparison

In [22]:
import os
import pandas as pd
import effspm

DATASET_DIR = "/Users/sandeep/Documents/library/effspm/datasets"

# Ground truth values from Table 3 of the JMLR paper
paper_truth = {
    ("MSNBC.txt", 0.05): 13,
    ("MSNBC.txt", 0.01): 254,
    ("MSNBC.txt", 0.005): 828,
    ("kosarak.txt", 0.01): 329,
    ("kosarak.txt", 0.005): 1462,
    ("kosarak.txt", 0.001): 758141,
}

# Testing all 6 exported algorithm variants
algorithms = ["HTMiner", "BTMiner", "PrefixProjection"]
rows = []

for (dataset, minsup), paper_count in paper_truth.items():
    ds_path = os.path.join(DATASET_DIR, dataset)
    row = {
        "Dataset": dataset,
        "Minsup": minsup,
        "Paper (Table 3)": paper_count
    }
    
    for algo in algorithms:
        try:
            miner_func = getattr(effspm, algo)
            res = miner_func(ds_path, minsup)
            # Safely extract pattern count from pybind11 dictionary
            count = len(res['patterns']) if isinstance(res, dict) and 'patterns' in res else len(res)
            row[f"{algo}"] = count
        except Exception as e:
            row[f"{algo}"] = f"Error"
            
    # Check if all tested algorithms match ground truth
    counts = [row[algo] for algo in algorithms if isinstance(row[algo], int)]
    row["All Match?"] = "PASS" if len(counts) == len(algorithms) and all(c == paper_count for c in counts) else "FAIL"
    
    rows.append(row)

# Display final comparison table
df_results = pd.DataFrame(rows)
df_results

,Dataset,Minsup,Paper (Table 3),HTMiner,BTMiner,PrefixProjection,All Match?
0,MSNBC.txt,0.050,13,13,13,13,PASS
1,MSNBC.txt,0.010,254,254,254,254,PASS
2,MSNBC.txt,0.005,828,828,828,828,PASS
3,kosarak.txt,0.010,329,329,329,329,PASS
4,kosarak.txt,0.005,1462,1462,1462,1462,PASS
5,kosarak.txt,0.001,758141,758141,758141,758141,PASS
